In [27]:
import re

In [28]:
# texto de exemplo
text = "Paul Newman was an American actor, but Paul Hollywood is a British TV Host. The name Paul is quite common."

# padrão para ser encontrado
pattern = r"Paul [A-Z]\w+"
# encontrado as correspondencias
matches = re.finditer(pattern, text)

# vendo as correspondencias
for match in matches:
    print (match)

<re.Match object; span=(0, 11), match='Paul Newman'>
<re.Match object; span=(39, 53), match='Paul Hollywood'>


In [29]:
import spacy
from spacy.tokens import Span

In [30]:
# criando pipeline vazio
nlp = spacy.blank('en')
# criando doc
doc = nlp(text)
# separando as entidades originais
original_ents = list(doc.ents)

mwt_ents = []
for match in re.finditer(pattern, doc.text):
    start, end = match.span()
    span = doc.char_span(start, end)
    if span is not None:
        mwt_ents.append((span.start, span.end, span.text))

for ent in mwt_ents:
    start, end, name = ent
    per_ent = Span(doc, start, end, label="PERSON")
    original_ents.append(per_ent)
doc.ents = original_ents
for ent in doc.ents:
    print(ent.text, ent.label_)

Paul Newman PERSON
Paul Hollywood PERSON


In [31]:
print(mwt_ents)

[(0, 2, 'Paul Newman'), (8, 10, 'Paul Hollywood')]


In [32]:
from spacy.language import Language

In [33]:
# definindo um componente
@Language.component('paul_ner')
def paul_ner(doc):

    # padrão para ser encontrado
    pattern = r"Paul [A-Z]\w+"
    # separando as entidades originais
    original_ents = list(doc.ents)

    mwt_ents = []
    for match in re.finditer(pattern, doc.text):
        start, end = match.span()
        span = doc.char_span(start, end)
        if span is not None:
            mwt_ents.append((span.start, span.end, span.text))

    for ent in mwt_ents:
        start, end, name = ent
        per_ent = Span(doc, start, end, label="PERSON")
        original_ents.append(per_ent)
    doc.ents = original_ents
    return doc

In [34]:
# pipeline vazio
nlp2 = spacy.blank('en')
# adicionando o componente no pipeline
nlp2.add_pipe('paul_ner')

<function __main__.paul_ner(doc)>

In [35]:
# criando um novo doc
doc2 = nlp2(text)
print(doc2.ents)

(Paul Newman, Paul Hollywood)


In [41]:
from spacy.language import Language
from spacy.util import filter_spans

# definindo um componente
@Language.component('cinema_ner')
def cinema_ner(doc):

    # padrão para ser encontrado
    pattern = r"Hollywood"
    # separando as entidades originais
    original_ents = list(doc.ents)

    mwt_ents = []
    for match in re.finditer(pattern, doc.text):
        start, end = match.span()
        span = doc.char_span(start, end)
        if span is not None:
            mwt_ents.append((span.start, span.end, span.text))

    for ent in mwt_ents:
        start, end, name = ent
        per_ent = Span(doc, start, end, label="CINEMA")
        original_ents.append(per_ent)
    fiiltered = filter_spans(original_ents)
    doc.ents = fiiltered
    return doc

In [42]:
# carregando o pipeline
nlp3 = spacy.load('en_core_web_sm')
nlp3.add_pipe('cinema_ner')

<function __main__.cinema_ner(doc)>

In [43]:
doc3 = nlp3(text)
for ent in doc3.ents:
    print(ent.text, ent.label_)

Paul Newman PERSON
American NORP
Paul Hollywood PERSON
British NORP
Paul PERSON
